In [1]:
from pathlib import Path

# Step up one folder from 'notebooks', then into 'data/raw'
data_dir = Path("../data/raw")

# Fallback if you just named it 'raw'
if not data_dir.exists():
    data_dir = Path("../raw")

print(f"Looking inside: {data_dir.resolve()}\n")

if data_dir.exists():
    print("--- Raw Data Files ---")
    for file_path in data_dir.glob("*"):
        if file_path.is_file():
            size_mb = file_path.stat().st_size / (1024 * 1024)
            print(f"{file_path.name} | Size: {size_mb:.2f} MB")
else:
    print("Folder still not found. Check your spelling in the file explorer.")

Looking inside: C:\Users\pc\Documents\Data Science Projects\dunnhumbychurn\data\raw

--- Raw Data Files ---
campaign_desc.csv | Size: 0.00 MB
campaign_table.csv | Size: 0.09 MB
causal_data.csv | Size: 663.62 MB
coupon.csv | Size: 2.69 MB
coupon_redempt.csv | Size: 0.05 MB
dunnhumby - The Complete Journey User Guide.pdf | Size: 0.93 MB
hh_demographic.csv | Size: 0.04 MB
product.csv | Size: 6.13 MB
transaction_data.csv | Size: 135.18 MB


In [2]:
!pip install duckdb

In [3]:
import duckdb

# Connect to a DuckDB database file in your processed folder.
# If it doesn't exist, DuckDB creates it automatically.
conn = duckdb.connect('../data/processed/dunnhumby.db')

# Use DuckDB's built-in CSV reader to query the file directly
query = """
SELECT *
FROM read_csv_auto('../data/raw/transaction_data.csv')
LIMIT 5;
"""

# Execute the SQL and convert the output to a Pandas DataFrame for clean formatting
df_head = conn.execute(query).df()
df_head


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [4]:
# Create the demographics table directly from the CSV
conn.execute("""
    CREATE TABLE IF NOT EXISTS demographics AS 
    SELECT * FROM read_csv_auto('../data/raw/hh_demographic.csv');
""")
conn.execute("SHOW TABLES;").df()

,name
0,demographics


In [6]:
# Create the transactions table directly from the CSV
conn.execute("""
    CREATE TABLE IF NOT EXISTS transactions AS 
    SELECT * FROM read_csv_auto('../data/raw/transaction_data.csv');
""")
conn.execute("SHOW TABLES;").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,name
0,demographics
1,transactions


In [7]:
# Create the product table directly from the CSV
conn.execute("""
    CREATE TABLE IF NOT EXISTS product AS 
    SELECT * FROM read_csv_auto('../data/raw/product.csv');
""")
conn.execute("SHOW TABLES;").df()

,name
0,demographics
1,product
2,transactions


In [8]:
# Query to find the first purchase day for each household
query_cohorts = """
SELECT 
    household_key,
    MIN(DAY) AS first_purchase_day
FROM transactions
GROUP BY household_key
LIMIT 5;
"""

df_cohorts = conn.execute(query_cohorts).df()
df_cohorts

,household_key,first_purchase_day
0,1,51
1,2,103
2,3,113
3,4,104
4,5,85


In [9]:
# Query to number each shopping trip chronologically
query_trips = """
WITH unique_baskets AS (
    -- Get one row per basket, keeping the day and time
    SELECT DISTINCT 
        household_key, 
        BASKET_ID, 
        DAY, 
        TRANS_TIME
    FROM transactions
)
SELECT 
    household_key,
    BASKET_ID,
    DAY,
    -- DENSE_RANK numbers the rows sequentially per household based on time
    DENSE_RANK() OVER (PARTITION BY household_key ORDER BY DAY, TRANS_TIME) AS trip_number
FROM unique_baskets
WHERE household_key = 1
ORDER BY trip_number
LIMIT 10;
"""

df_trips = conn.execute(query_trips).df()
df_trips

,household_key,BASKET_ID,DAY,trip_number
0,1,27601281299,51,1
1,1,27774192959,67,2
2,1,28024266849,88,3
3,1,28106322445,94,4
4,1,28235481967,101,5
5,1,28282581446,108,6
6,1,28318295376,111,7
7,1,28601601153,128,8
8,1,28744672720,137,9
9,1,28853646373,146,10


In [10]:
query_funnel = """
WITH unique_baskets AS (
    SELECT DISTINCT household_key, BASKET_ID, DAY, TRANS_TIME
    FROM transactions
),
numbered_trips AS (
    SELECT 
        household_key,
        DENSE_RANK() OVER (PARTITION BY household_key ORDER BY DAY, TRANS_TIME) AS trip_number
    FROM unique_baskets
)
-- Your turn: Write the final SELECT to count unique households per trip_number
SELECT 
    trip_number,
    COUNT(DISTINCT household_key) AS household_count
FROM numbered_trips
WHERE trip_number <= 10
GROUP BY trip_number
ORDER BY trip_number;
"""

df_funnel = conn.execute(query_funnel).df()
df_funnel

,trip_number,household_count
0,1,2500
1,2,2497
2,3,2492
3,4,2483
4,5,2477
5,6,2466
6,7,2448
7,8,2438
8,9,2424
9,10,2402


In [11]:
# Calculate the survival rate as a percentage of the original 2,500 households
df_funnel['survival_rate_pct'] = (df_funnel['household_count'] / df_funnel['household_count'].max()) * 100

# Display the final funnel
df_funnel.round(2)

,trip_number,household_count,survival_rate_pct
0,1,2500,100.00
1,2,2497,99.88
2,3,2492,99.68
3,4,2483,99.32
4,5,2477,99.08
5,6,2466,98.64
6,7,2448,97.92
7,8,2438,97.52
8,9,2424,96.96
9,10,2402,96.08


In [16]:
# Query to calculate Recency, Frequency, and Monetary value per household
query_rfm = """
WITH dataset_max AS (
    SELECT MAX(DAY) AS current_day 
    FROM transactions
)
SELECT 
    household_key,
    (SELECT current_day FROM dataset_max) - MAX(DAY) AS recency,
    
  COUNT(DISTINCT BASKET_ID) AS frequency,
    
  SUM(SALES_VALUE) AS monetary
    
FROM transactions
GROUP BY household_key
ORDER BY monetary DESC
LIMIT 5;
"""

df_rfm = conn.execute(query_rfm).df()
df_rfm

,household_key,recency,frequency,monetary
0,1023,1,603,38319.79
1,1609,0,412,27859.68
2,2322,0,323,23646.92
3,1453,1,761,21661.29
4,2459,7,971,20671.50


In [33]:
query_segments = """
WITH rfm_raw AS (
    WITH dataset_max AS (SELECT MAX(DAY) AS current_day FROM transactions)
    SELECT 
        household_key,
        (SELECT current_day FROM dataset_max) - MAX(DAY) AS recency,
        COUNT(DISTINCT BASKET_ID) AS frequency,
        SUM(SALES_VALUE) AS monetary
    FROM transactions
    GROUP BY household_key
),
rfm_scores AS (
    SELECT 
        household_key,
        recency,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm_raw
)
SELECT 
    household_key,
    r_score,
    f_score,
    m_score,
    
    -- Your turn: Use CONCAT() to combine r_score, f_score, and m_score into a column named rfm_class
    
    
FROM rfm_scores
ORDER BY monetary DESC;
"""

df_segments = conn.execute(query_segments).df()
df_segments

,household_key,r_score,f_score,m_score
0,1023,4,5,5
1,1609,5,5,5
2,2322,5,5,5
3,1453,5,5,5
4,2459,3,5,5
...,...,...,...,...
2495,522,1,1,1
2496,689,1,1,1
2497,2201,1,1,1
2498,1897,1,1,1


In [32]:
query_segments = """
WITH rfm_raw AS (
    WITH dataset_max AS (SELECT MAX(DAY) AS current_day FROM transactions)
    SELECT 
        household_key,
        (SELECT current_day FROM dataset_max) - MAX(DAY) AS recency,
        COUNT(DISTINCT BASKET_ID) AS frequency,
        SUM(SALES_VALUE) AS monetary
    FROM transactions
    GROUP BY household_key
),
rfm_scores AS (
    SELECT 
        household_key,
        recency,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm_raw
)
SELECT 
    household_key,
    r_score,
    f_score,
    m_score,
    CONCAT(r_score, f_score, m_score) AS rfm_class
FROM rfm_scores
ORDER BY monetary DESC;
"""

df_segments = conn.execute(query_segments).df()
df_segments

,household_key,r_score,f_score,m_score,rfm_class
0,1023,4,5,5,455
1,1609,5,5,5,555
2,2322,5,5,5,555
3,1453,4,5,5,455
4,2459,3,5,5,355
...,...,...,...,...,...
2495,522,1,1,1,111
2496,689,1,1,1,111
2497,2201,1,1,1,111
2498,1897,1,1,1,111


In [24]:
# 1. Pull the chronological shopping days for every unique basket
query_gaps = """
SELECT 
    household_key, 
    BASKET_ID, 
    DAY
FROM transactions
GROUP BY household_key, BASKET_ID, DAY
ORDER BY household_key, DAY;
"""
df_gaps = conn.execute(query_gaps).df()

# 2. Calculate the days between consecutive trips for each household
df_gaps['days_between_trips'] = df_gaps.groupby('household_key')['DAY'].diff()

# 3. Look at the distribution of these gaps
print(" Distribution of Days Between Shopping Trips ")
print(df_gaps['days_between_trips'].describe())

 Distribution of Days Between Shopping Trips 
count    273984.000000
mean          5.652564
std          14.571735
min           0.000000
25%           1.000000
50%           2.000000
75%           6.000000
max         666.000000
Name: days_between_trips, dtype: float64


In [30]:
query_ml_data = """
WITH cutoff_calc AS (
    SELECT MAX(DAY) - 60 AS cutoff_day FROM transactions
),
churn_labels AS (
    SELECT 
        household_key,
        CASE WHEN MAX(DAY) > (SELECT cutoff_day FROM cutoff_calc) THEN 0 ELSE 1 END AS churn_label
    FROM transactions
    GROUP BY household_key
),
pre_cutoff_rfm AS (
    SELECT 
        household_key,
        (SELECT cutoff_day FROM cutoff_calc) - MAX(DAY) AS recency,
        COUNT(DISTINCT BASKET_ID) AS frequency,
        SUM(SALES_VALUE) AS monetary
    FROM transactions
    -- STRICT TIME SPLIT: Only look at data before the cutoff
    WHERE DAY <= (SELECT cutoff_day FROM cutoff_calc)
    GROUP BY household_key
)
SELECT 
    l.household_key,
    l.churn_label,
    r.recency,
    r.frequency,
    r.monetary
FROM churn_labels l
JOIN pre_cutoff_rfm r ON l.household_key = r.household_key
ORDER BY l.household_key;
"""

df_model = conn.execute(query_ml_data).df()
df_model

,household_key,churn_label,recency,frequency,monetary
0,1,0,2,78,3869.70
1,2,0,29,44,1823.45
2,3,0,11,45,2594.30
3,4,1,24,30,1200.11
4,5,0,62,38,749.09
...,...,...,...,...,...
2494,2496,0,17,59,4095.29
2495,2497,0,1,210,6503.87
2496,2498,0,1,155,2481.26
2497,2499,0,10,85,2874.39


In [31]:
# Check the absolute counts
print("Target Variable Distribution")
print(df_model['churn_label'].value_counts())

# Check the percentage breakdown (baseline churn rate)
print("\n Baseline Churn Rate ")
print(df_model['churn_label'].value_counts(normalize=True).round(4) * 100)

Target Variable Distribution
churn_label
0    2238
1     261
Name: count, dtype: int64

 Baseline Churn Rate 
churn_label
0    89.56
1    10.44
Name: proportion, dtype: float64


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Define features (X) and target (y)

X = df_model[['recency', 'frequency', 'monetary']]
y = df_model['churn_label']

# 2. Time-aware strict split (stratified to keep the 10% class balance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Initialize and train the Random Forest
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# 4. Predict on the unseen test set
y_pred = rf_model.predict(X_test)

# 5. Evaluate
print("Baseline Model Evaluation")
print(classification_report(y_test, y_pred))

Baseline Model Evaluation
              precision    recall  f1-score   support

           0       0.94      0.90      0.92       448
           1       0.37      0.50      0.42        52

    accuracy                           0.86       500
   macro avg       0.65      0.70      0.67       500
weighted avg       0.88      0.86      0.87       500



In [38]:
import pandas as pd

# Extract feature importances from the Random Forest
importances = rf_model.feature_importances_

# Create a clean table to view the rankings
df_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("What Drives Household Churn?")
print(df_importance)

What Drives Household Churn?
     Feature  Importance
0    recency    0.394181
2   monetary    0.305047
1  frequency    0.300772


In [39]:
import os

# Ensure the processed folder exists
os.makedirs('../data/processed', exist_ok=True)

# Save your master ML dataset into the processed folder
df_model.to_csv('../data/processed/master_churn_model_data.csv', index=False)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
